# Universal Player Classification (Outfield & Goalkeepers).

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load datasets
df_outfield = pd.read_csv('../data/clean/clean_df_outfield.csv')
df_gk = pd.read_csv('../data/clean/clean_df_gk.csv')
print(f"Total outfield players loaded: {len(df_outfield)}")
print(f"Total goalkeepers loaded: {len(df_gk)}")

Total outfield players loaded: 772
Total goalkeepers loaded: 52


In [2]:
import importlib
import scripts.classify
import scripts.config

In [3]:
from scripts.classify import *
from scripts.config import *

In [4]:
final_outfield_df = classify_players(df_outfield, outfield_config, hybrid_threshold=5.0)
final_gk_df       = classify_players(df_gk,       gk_config,       hybrid_threshold=5.0)

final_outfield_df.to_csv('outfield_classified.csv', index=False)
final_gk_df.to_csv('gk_classified.csv', index=False)

In [5]:
print("Outfield Tactical Role Distribution:\n")
print(final_outfield_df.groupby(['Broad_Position', 'Tactical_Role']).size().unstack(fill_value=0))
print("\n" + "="*70 + "\n")

print("Goalkeeper Tactical Role Distribution:\n")
print(final_gk_df.groupby(['Broad_Position', 'Tactical_Role']).size().unstack(fill_value=0))
print("\n" + "="*70 + "\n")

Outfield Tactical Role Distribution:

Tactical_Role       Aerial Stopper  Aerial Stopper / Ball Playing Defender  \
Broad_Position                                                               
Attacking Midfield               0                                       0   
Central Midfield                 0                                       0   
Centre-Back                     41                                       2   
Defensive Midfield               0                                       0   
Full-Back                        0                                       0   
Striker                          0                                       0   
Winger                           0                                       0   

Tactical_Role       Aerial Stopper / Defensive Sweeper  Anchor Man  \
Broad_Position                                                       
Attacking Midfield                                   0           0   
Central Midfield                                 

In [6]:
final_outfield_df.to_csv('../data/clean/classified_outfield_players.csv', index=False)
final_gk_df.to_csv('../data/clean/classified_goalkeepers.csv', index=False)
print("Data successfully exported to 'classified_outfield_players.csv' and 'classified_goalkeepers.csv'.")

Data successfully exported to 'classified_outfield_players.csv' and 'classified_goalkeepers.csv'.


# CLASSIFY PLAYERS INTO 7 POSITION CATEGORIES

In [46]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [47]:
# Load your classified data
df = pd.read_csv('../data/clean/classified_outfield_players.csv')

# Display current broad positions
print("Current Broad_Position categories:")
print(df['Broad_Position'].unique())
print(f"\nTotal players: {len(df)}")

Current Broad_Position categories:
<StringArray>
[       'Centre-Back',          'Full-Back', 'Defensive Midfield',
   'Central Midfield', 'Attacking Midfield',             'Winger',
            'Striker']
Length: 7, dtype: str

Total players: 772


In [50]:
def classify_position_7(broad_position, position):
    """
    Classify players into 7 standardized position categories:
    - Centre-Back
    - Full-Back (includes Left-Back and Right-Back - keeps original distinction)
    - Defensive Midfield
    - Central Midfield
    - Attacking Midfield
    - Winger (includes Left Winger and Right Winger)
    - Striker (includes Centre-Forward and Second Striker)
    """
    
    # Centre-Back
    if broad_position == 'Centre-Back' or position == 'Centre-Back':
        return 'Centre-Back'
    
    # Full-Back (includes Left-Back and Right-Back)
    # We keep the original position name in a separate column
    if broad_position == 'Full-Back' or position in ['Left-Back', 'Right-Back']:
        return 'Full-Back'
    
    # Defensive Midfield
    if broad_position == 'Defensive Midfield':
        return 'Defensive Midfield'
    
    # Central Midfield
    if broad_position == 'Central Midfield':
        return 'Central Midfield'
    
    # Attacking Midfield
    if broad_position == 'Attacking Midfield':
        return 'Attacking Midfield'
    
    # Winger (includes Left Winger and Right Winger)
    if broad_position == 'Winger' or position in ['Left Winger', 'Right Winger']:
        return 'Winger'
    
    # Striker (includes Centre-Forward and Second Striker)
    if broad_position == 'Striker' or position in ['Centre-Forward', 'Second Striker']:
        return 'Striker'
    
    # Default - should not happen
    return 'Other'

# Apply classification
df['Position_7'] = df.apply(
    lambda row: classify_position_7(row['Broad_Position'], row['Position']), 
    axis=1
)

# ============================================
# CREATE FULL-BACK SUB-CATEGORY COLUMN
# ============================================

def get_fullback_type(row):
    """
    Determine if a full-back is Left-Back or Right-Back
    """
    if row['Position_7'] == 'Full-Back':
        if 'Left-Back' in str(row['Position']):
            return 'Left-Back'
        elif 'Right-Back' in str(row['Position']):
            return 'Right-Back'
        elif 'Left' in str(row['Broad_Position']):
            return 'Left-Back'
        elif 'Right' in str(row['Broad_Position']):
            return 'Right-Back'
        else:
            return 'Full-Back (Unspecified)'
    return None

# Add full-back specific column
df['FullBack_Type'] = df.apply(get_fullback_type, axis=1)

# Also add winger sub-category for consistency
def get_winger_type(row):
    """
    Determine if a winger is Left Winger or Right Winger
    """
    if row['Position_7'] == 'Winger':
        if 'Left Winger' in str(row['Position']):
            return 'Left Winger'
        elif 'Right Winger' in str(row['Position']):
            return 'Right Winger'
        elif 'Left' in str(row['Broad_Position']):
            return 'Left Winger'
        elif 'Right' in str(row['Broad_Position']):
            return 'Right Winger'
        else:
            return 'Winger (Unspecified)'
    return None

df['Winger_Type'] = df.apply(get_winger_type, axis=1)

In [39]:
# ============================================
# VERIFY CLASSIFICATION
# ============================================

In [52]:
print("\n" + "="*70)
print("7 POSITION CATEGORIES DISTRIBUTION")
print("="*70)
position_counts = df['Position_7'].value_counts()
print(position_counts)
print(f"\nTotal: {position_counts.sum()} players")

# Show Full-Back breakdown
print("\n" + "="*70)
print("FULL-BACK BREAKDOWN")
print("="*70)
fullback_breakdown = df[df['Position_7'] == 'Full-Back']['FullBack_Type'].value_counts()
print(fullback_breakdown)

# Show Winger breakdown
print("\n" + "="*70)
print("WINGER BREAKDOWN")
print("="*70)
winger_breakdown = df[df['Position_7'] == 'Winger']['Winger_Type'].value_counts()
print(winger_breakdown)

# Check if any "Other" category exists
if 'Other' in position_counts.index:
    print("\nWARNING: 'Other' category found!")
    print(df[df['Position_7'] == 'Other'][['Name', 'Broad_Position', 'Position']].head())


7 POSITION CATEGORIES DISTRIBUTION
Position_7
Centre-Back           149
Full-Back             145
Winger                130
Central Midfield      115
Striker               111
Defensive Midfield     68
Attacking Midfield     54
Name: count, dtype: int64

Total: 772 players

FULL-BACK BREAKDOWN
FullBack_Type
Right-Back    77
Left-Back     68
Name: count, dtype: int64

WINGER BREAKDOWN
Winger_Type
Right Winger            66
Left Winger             62
Winger (Unspecified)     2
Name: count, dtype: int64


In [53]:
# ============================================
# FUNCTION TO REMOVE NULL COLUMNS AFTER SPLITTING
# ============================================

In [60]:
def clean_position_table(df, position_name):
    """
    Remove columns that are completely null (100% missing) for a specific position
    Also removes columns that are constant (all same value) and rows that are completely empty
    Returns: (cleaned_dataframe, list_of_removed_columns)
    """
    original_rows = len(df)
    original_cols = len(df.columns)
    
    # Make a copy to avoid modifying original
    df_clean = df.copy()
    
    # Keep track of all removed columns
    all_removed = []
    
    # Step 1: Remove columns that are 100% null
    cols_all_null = df_clean.columns[df_clean.isna().all()].tolist()
    all_removed.extend(cols_all_null)
    df_clean = df_clean.drop(columns=cols_all_null)
    
    # Step 2: Remove constant columns (all same value, excluding nulls)
    constant_cols = []
    for col in df_clean.columns:
        # Skip columns that should keep their distinction
        if col in ['FullBack_Type', 'Winger_Type', 'Position']:
            continue
        # Get unique non-null values
        unique_vals = df_clean[col].dropna().unique()
        # If only one unique value (or all nulls), it's constant
        if len(unique_vals) <= 1:
            constant_cols.append(col)
    all_removed.extend(constant_cols)
    df_clean = df_clean.drop(columns=constant_cols)
    
    # Step 3: Remove rows that are completely empty
    before_rows = len(df_clean)
    df_clean = df_clean.dropna(how='all')
    rows_removed = before_rows - len(df_clean)
    
    # Step 4: Remove columns that are now 100% null after row removal (safety check)
    cols_all_null_after = df_clean.columns[df_clean.isna().all()].tolist()
    all_removed.extend(cols_all_null_after)
    df_clean = df_clean.drop(columns=cols_all_null_after)
    
    # Print cleaning report
    print(f"\n{position_name}:")
    print(f"   Original: {original_rows} rows, {original_cols} columns")
    print(f"   Removed 100% null columns: {len(cols_all_null)}")
    print(f"   Removed constant columns: {len(constant_cols)}")
    print(f"   Removed {rows_removed} empty rows")
    print(f"   After cleaning: {len(df_clean)} rows, {len(df_clean.columns)} columns")
    print(f"   Total columns removed: {original_cols - len(df_clean.columns)}")
    
    # Show sample of removed columns
    if all_removed and len(all_removed) <= 20:
        print(f"   Removed columns: {all_removed}")
    elif all_removed:
        print(f"   Sample removed columns: {all_removed[:10]}{'...' if len(all_removed) > 10 else ''}")
    
    # Return both the cleaned dataframe AND the list of removed columns
    return df_clean, all_removed

In [61]:
# ============================================
# SPLIT DATA BY POSITION (BEFORE CLEANING)
# ============================================

In [62]:
# Create raw position dataframes (before cleaning)
centre_backs_raw = df[df['Position_7'] == 'Centre-Back'].copy()
full_backs_raw = df[df['Position_7'] == 'Full-Back'].copy()
defensive_midfielders_raw = df[df['Position_7'] == 'Defensive Midfield'].copy()
central_midfielders_raw = df[df['Position_7'] == 'Central Midfield'].copy()
attacking_midfielders_raw = df[df['Position_7'] == 'Attacking Midfield'].copy()
wingers_raw = df[df['Position_7'] == 'Winger'].copy()
strikers_raw = df[df['Position_7'] == 'Striker'].copy()

print(f"Centre-Backs:          {len(centre_backs_raw)} players, {len(centre_backs_raw.columns)} columns")
print(f"Full-Backs:            {len(full_backs_raw)} players, {len(full_backs_raw.columns)} columns")
print(f"  - Left-Backs:        {len(full_backs_raw[full_backs_raw['FullBack_Type'] == 'Left-Back'])}")
print(f"  - Right-Backs:       {len(full_backs_raw[full_backs_raw['FullBack_Type'] == 'Right-Back'])}")
print(f"Defensive Midfielders: {len(defensive_midfielders_raw)} players, {len(defensive_midfielders_raw.columns)} columns")
print(f"Central Midfielders:   {len(central_midfielders_raw)} players, {len(central_midfielders_raw.columns)} columns")
print(f"Attacking Midfielders: {len(attacking_midfielders_raw)} players, {len(attacking_midfielders_raw.columns)} columns")
print(f"Wingers:               {len(wingers_raw)} players, {len(wingers_raw.columns)} columns")
print(f"  - Left Wingers:      {len(wingers_raw[wingers_raw['Winger_Type'] == 'Left Winger'])}")
print(f"  - Right Wingers:     {len(wingers_raw[wingers_raw['Winger_Type'] == 'Right Winger'])}")
print(f"Strikers:              {len(strikers_raw)} players, {len(strikers_raw.columns)} columns")

Centre-Backs:          149 players, 107 columns
Full-Backs:            145 players, 107 columns
  - Left-Backs:        68
  - Right-Backs:       77
Defensive Midfielders: 68 players, 107 columns
Central Midfielders:   115 players, 107 columns
Attacking Midfielders: 54 players, 107 columns
Wingers:               130 players, 107 columns
  - Left Wingers:      62
  - Right Wingers:     66
Strikers:              111 players, 107 columns


In [63]:
# ============================================
# CLEAN EACH POSITION TABLE (REMOVE NULL COLUMNS)
# ============================================

In [65]:
print("\n" + "="*70)
print("CLEANING NULL COLUMNS FOR EACH POSITION")
print("="*70)

centre_backs, cb_removed = clean_position_table(centre_backs_raw, "Centre-Backs")
full_backs, fb_removed = clean_position_table(full_backs_raw, "Full-Backs")
defensive_midfielders, dm_removed = clean_position_table(defensive_midfielders_raw, "Defensive Midfielders")
central_midfielders, cm_removed = clean_position_table(central_midfielders_raw, "Central Midfielders")
attacking_midfielders, am_removed = clean_position_table(attacking_midfielders_raw, "Attacking Midfielders")
wingers, w_removed = clean_position_table(wingers_raw, "Wingers")
strikers, s_removed = clean_position_table(strikers_raw, "Strikers")


CLEANING NULL COLUMNS FOR EACH POSITION

Centre-Backs:
   Original: 149 rows, 107 columns
   Removed 100% null columns: 16
   Removed constant columns: 5
   Removed 0 empty rows
   After cleaning: 149 rows, 86 columns
   Total columns removed: 21
   Sample removed columns: ['Wing Back_Score', 'Inverted Full Back_Score', 'Defensive Full Back_Score', 'Anchor Man_Score', 'Deep Lying Playmaker_Score', 'Ball Winning Midfielder_Score', 'Mezzala_Score', 'Shadow Striker_Score', 'Playmaker_Score', 'Inside Forward_Score']...

Full-Backs:
   Original: 145 rows, 107 columns
   Removed 100% null columns: 15
   Removed constant columns: 5
   Removed 0 empty rows
   After cleaning: 145 rows, 87 columns
   Total columns removed: 20
   Removed columns: ['Aerial Stopper_Score', 'Ball Playing Defender_Score', 'Defensive Sweeper_Score', 'Anchor Man_Score', 'Deep Lying Playmaker_Score', 'Ball Winning Midfielder_Score', 'Mezzala_Score', 'Shadow Striker_Score', 'Playmaker_Score', 'Inside Forward_Score', 'Tr

In [66]:
# ============================================
# SAVE THE CLASSIFIED & CLEANED DATAFRAMES
# ============================================

In [67]:
# Save main dataframe with new Position_7 column
df.to_csv('../data/clean/classified_outfield_position_7.csv', index=False)

# Save cleaned separate position dataframes
centre_backs.to_csv('../data/clean/position_centre_backs_clean.csv', index=False)
full_backs.to_csv('../data/clean/position_full_backs_clean.csv', index=False)
defensive_midfielders.to_csv('../data/clean/position_defensive_midfielders_clean.csv', index=False)
central_midfielders.to_csv('../data/clean/position_central_midfielders_clean.csv', index=False)
attacking_midfielders.to_csv('../data/clean/position_attacking_midfielders_clean.csv', index=False)
wingers.to_csv('../data/clean/position_wingers_clean.csv', index=False)
strikers.to_csv('../data/clean/position_strikers_clean.csv', index=False)

print("\n" + "="*70)
print("✅ CLASSIFICATION & CLEANING COMPLETE! Files saved:")
print("="*70)
print("Main file:")
print("   - classified_outfield_position_7.csv (with Position_7 column)")
print("\nCleaned position files (null columns removed per position):")
print("   - position_centre_backs_clean.csv")
print("   - position_full_backs_clean.csv (includes Left-Back and Right-Back distinction)")
print("   - position_defensive_midfielders_clean.csv")
print("   - position_central_midfielders_clean.csv")
print("   - position_attacking_midfielders_clean.csv")
print("   - position_wingers_clean.csv (includes Left Winger and Right Winger distinction)")
print("   - position_strikers_clean.csv")
print("="*70)


✅ CLASSIFICATION & CLEANING COMPLETE! Files saved:
Main file:
   - classified_outfield_position_7.csv (with Position_7 column)

Cleaned position files (null columns removed per position):
   - position_centre_backs_clean.csv
   - position_full_backs_clean.csv (includes Left-Back and Right-Back distinction)
   - position_defensive_midfielders_clean.csv
   - position_central_midfielders_clean.csv
   - position_attacking_midfielders_clean.csv
   - position_wingers_clean.csv (includes Left Winger and Right Winger distinction)
   - position_strikers_clean.csv
